# Phase 3: Supervised Learning for False Positive Reduction
## Task 3.4: Neural Network (Multi-Layer Perceptron)

Build and tune a supervised flow-level neural network that separates benign and attack flows.

This notebook uses the **same fixed Phase 3 flow splits** as the Random Forest, XGBoost, and SVM
notebooks so all four Stage-2 classifiers can be compared fairly. It follows the same structure:
load the splits, preprocess, tune with cross-validation on the training split only, then evaluate
once on the untouched validation and test sets.

**Why a neural network here.** A multi-layer perceptron (MLP) learns non-linear decision boundaries
through hidden layers, which can capture feature interactions that a single linear boundary (SVM) or
axis-aligned tree splits (Random Forest / XGBoost) approximate differently. The point of training it
is not to win outright but to give the team a fourth, independent Stage-2 candidate whose behaviour
can be compared on identical data.

Author: Anand, Kuldeep, Priyansh

## Setup and load the fixed flow splits

In [ ]:
from sklearn.neural_network import MLPClassifier

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    average_precision_score,
    precision_recall_curve,
    make_scorer
)

def find_project_root(start):
    """Walk up from `start` until the folder containing src/data/flow_preprocessing.py is found.

    This makes the notebook run correctly no matter which working directory the kernel uses
    (notebook folder or workspace root), as long as the notebook lives inside the cloned repo.
    """
    marker = Path("src") / "data" / "flow_preprocessing.py"
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root (no src/data/flow_preprocessing.py found above "
        f"{start}). Place this notebook inside the multi-stage-iot-ids repo under notebooks/ "
        "and run it from there."
    )

# In a notebook __file__ is not defined, so start the search from the current working directory.
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

from src.data.flow_preprocessing import ATTACK_COL, BINARY_COL, FLOW_KEY_COL, TARGET_COL, build_flow_preprocessor

DATA_FOLDER = "phase3"
DATA_DIR = PROJECT_ROOT / "data" / "processed" / DATA_FOLDER
META_PATH = DATA_DIR / "flow_split_metadata.json"

with open(META_PATH, "r") as f:
    metadata = json.load(f)

train_df = pd.read_csv(DATA_DIR / "flow_train.csv", low_memory=False)
validation_df = pd.read_csv(DATA_DIR / "flow_validation.csv", low_memory=False)
test_df = pd.read_csv(DATA_DIR / "flow_test.csv", low_memory=False)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)
metadata

### Verify the class distribution and split integrity

In [ ]:
for name, current_df in [("Train", train_df), ("Validation", validation_df), ("Test", test_df)]:
    print(f"{name} binary counts")
    print(current_df[BINARY_COL].value_counts())
    print()

train_keys = set(train_df[FLOW_KEY_COL])
validation_keys = set(validation_df[FLOW_KEY_COL])
test_keys = set(test_df[FLOW_KEY_COL])

print("Flow-key overlap")
print("  train n validation:", len(train_keys & validation_keys))
print("  train n test:", len(train_keys & test_keys))
print("  validation n test:", len(validation_keys & test_keys))

## Neural network feature preparation

The model uses the numeric flow-behaviour features recorded in the split metadata. Labels, flow
identifiers, IP addresses, timestamps, and source-file information are excluded.

Unlike the tree models, a neural network is **sensitive to feature scale**: features with large raw
ranges would dominate the gradient and the network would train poorly. The shared preprocessor is
therefore called with `scale=True`, which appends a `RobustScaler` after median imputation and
constant-column removal. `RobustScaler` centres on the median and scales by the interquartile range,
so the heavy-tailed network-traffic features are standardised without letting extreme outliers
distort the scale.

In [ ]:
feature_cols = metadata["feature_columns"]
missing_features = [col for col in feature_cols if col not in train_df.columns]

if missing_features:
    raise ValueError(f"Missing model features: {missing_features}")

X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].to_numpy()

X_validation = validation_df[feature_cols]
y_validation = validation_df[TARGET_COL].to_numpy()

X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL].to_numpy()

print("Input features:", len(feature_cols))
print("Training missing values:", int(X_train.isna().sum().sum()))
print("Validation missing values:", int(X_validation.isna().sum().sum()))
print("Test missing values:", int(X_test.isna().sum().sum()))

## Handle class imbalance on the training split only

The flow data is heavily imbalanced: benign flows vastly outnumber attack flows. The tree models
handle this with a class weight (`scale_pos_weight` for XGBoost, `class_weight="balanced"` for the
SVM). `MLPClassifier` supports neither class weights nor per-sample weights, so we rebalance the
**training data** directly — but a neural network needs *data*, so the key is to keep as much of it as
possible:

1. **Keep a large real benign sample** (up to `MAX_BENIGN_TRAIN`), not a tiny one. An earlier version
   undersampled benign down to a few times the attack count, which threw away ~90% of the data and
   starved the network — that is the main reason its F1 lagged the Random Forest. Keeping tens of
   thousands of benign rows gives the model far more signal about what normal traffic looks like,
   which is what lifts precision and PR-AUC.
2. **Oversample the attack rows** (sampling with replacement) up to `ATTACK_TO_BENIGN` times the kept
   benign count. This is the scikit-learn-only stand-in for a class weight: duplicating minority rows
   makes the network pay proportionally more attention to attacks, without discarding benign data.

Only the **training** split is touched. Validation and test keep their true, imbalanced distribution,
so every reported metric reflects reality. `MAX_BENIGN_TRAIN` also bounds run time — raise it for a
stronger (slower) model, lower it for a quicker run.

In [ ]:
RANDOM_STATE = 1
MAX_BENIGN_TRAIN = 20000      # keep up to this many real benign rows (was a tiny ~3x-attack sample)
ATTACK_TO_BENIGN = 0.6        # after oversampling, attacks reach this fraction of the kept benign

rng = np.random.RandomState(RANDOM_STATE)

benign_idx = np.where(y_train == 0)[0]
attack_idx = np.where(y_train == 1)[0]

# 1) keep a large sample of real benign rows (capped only for run time)
n_benign_keep = min(len(benign_idx), MAX_BENIGN_TRAIN)
benign_keep = rng.choice(benign_idx, size=n_benign_keep, replace=False)

# 2) oversample the attack rows with replacement up to the target count
n_attack_target = max(int(ATTACK_TO_BENIGN * n_benign_keep), len(attack_idx))
attack_resampled = rng.choice(attack_idx, size=n_attack_target, replace=True)

keep_idx = np.concatenate([benign_keep, attack_resampled])
rng.shuffle(keep_idx)

X_train_balanced = X_train.iloc[keep_idx].reset_index(drop=True)
y_train_balanced = y_train[keep_idx]

print("Original training rows:", len(y_train),
      "| benign:", len(benign_idx), " attack:", len(attack_idx))
print("Resampled training rows:", len(y_train_balanced),
      "| benign:", int((y_train_balanced == 0).sum()),
      " attack:", int((y_train_balanced == 1).sum()),
      f"({len(attack_idx)} unique attacks oversampled to {n_attack_target})")

## Tune the MLP with stratified cross-validation

**How the tuning works — two tools, one job.** `RandomizedSearchCV` is the *search*: it samples
12 random hyperparameter combinations. `StratifiedKFold` is the *scoring method it uses inside*: for
each of those 12 combinations it trains and scores across **5 folds** (each fold keeps the same
benign-to-attack ratio) and averages. So both are used together — the search picks what to try, k-fold
scores how good each try is. The search runs on the rebalanced training set and is refit on **PR-AUC**,
the average-precision metric that stays informative under class imbalance.

The network is an Adam-optimised MLP with **early stopping**: it holds out a small internal validation
fraction and stops training when the validation score stops improving, which prevents overfitting and
keeps run time bounded. The search explores network width and depth (`hidden_layer_sizes`), L2
regularisation strength (`alpha`), the initial learning rate, the activation function, and the
mini-batch size.

In [ ]:
N_ITER = 12
DEFAULT_THRESHOLD = 0.50

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

preprocessor = build_flow_preprocessor(scale=True)
mlp_model = MLPClassifier(
    solver="adam",
    early_stopping=True,
    n_iter_no_change=12,
    validation_fraction=0.1,
    max_iter=500,
    random_state=RANDOM_STATE
)

mlp_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", mlp_model)
])

param_distributions = {
    "model__hidden_layer_sizes": [(128,), (256,), (128, 64), (256, 128), (256, 128, 64)],
    "model__activation": ["relu", "tanh"],
    "model__alpha": [1e-4, 1e-3, 1e-2, 1e-1],
    "model__learning_rate_init": [5e-4, 1e-3, 2e-3],
    "model__batch_size": [256, 512]
}

scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1": make_scorer(f1_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0)
}

mlp_search = RandomizedSearchCV(
    estimator=mlp_pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=RANDOM_STATE,
    return_train_score=False
)

start_search = time.perf_counter()
mlp_search.fit(X_train_balanced, y_train_balanced)
search_time = time.perf_counter() - start_search

mlp_pipeline = mlp_search.best_estimator_

print(f"Search time: {search_time:.2f} seconds")
print(f"Best mean CV PR-AUC: {mlp_search.best_score_:.4f}")
print("Best parameters:")
for name, value in mlp_search.best_params_.items():
    print(f"  {name}: {value}")

processed_feature_count = mlp_pipeline.named_steps["preprocessor"].named_steps["remove_constant"].get_support().sum()
print("Processed features:", processed_feature_count)

### Cross-validation results

The table below shows the highest-ranked sampled configurations. The final estimator is the row with
rank 1 for mean validation PR-AUC across the folds.

In [ ]:
cv_results_df = pd.DataFrame(mlp_search.cv_results_)

result_columns = [
    "rank_test_pr_auc",
    "mean_test_pr_auc",
    "std_test_pr_auc",
    "mean_test_roc_auc",
    "mean_test_f1",
    "mean_test_precision",
    "mean_test_recall",
    "mean_fit_time",
    "param_model__hidden_layer_sizes",
    "param_model__activation",
    "param_model__alpha",
    "param_model__learning_rate_init",
    "param_model__batch_size"
]

top_cv_results = cv_results_df[result_columns].sort_values("rank_test_pr_auc").head(10)
display(top_cv_results.round(4))

## Choose the decision threshold that maximises F1

**This is the main lever for F1.** A probability model does not have to decide at 0.5 — that
cut-off is arbitrary, and because we undersampled the benign class during training, the model's scores
are shifted and 0.5 is usually *not* the best operating point. So after the model is fixed, we sweep
the decision threshold on the **validation set** and pick the value that maximises F1, then apply that
single threshold, unchanged, to the test set.

This is a standard and honest step: the model itself is never re-trained on the test data, and the
threshold is chosen only from validation. It is what lets the network trade a little recall for
precision (or vice-versa) to land the best F1, so it can compete with the Random Forest baseline.

In [ ]:
validation_scores = mlp_pipeline.predict_proba(X_validation)[:, 1]

candidate_thresholds = np.linspace(0.05, 0.95, 181)
f1_at_threshold = [
    (t, f1_score(y_validation, (validation_scores >= t).astype(int), zero_division=0))
    for t in candidate_thresholds
]
tuned_threshold, tuned_validation_f1 = max(f1_at_threshold, key=lambda pair: pair[1])
default_validation_f1 = f1_score(y_validation, (validation_scores >= DEFAULT_THRESHOLD).astype(int), zero_division=0)

print(f"Default threshold {DEFAULT_THRESHOLD:.2f}  ->  validation F1 = {default_validation_f1:.4f}")
print(f"Tuned threshold   {tuned_threshold:.3f}  ->  validation F1 = {tuned_validation_f1:.4f}")
print(f"Validation F1 gain from threshold tuning: {tuned_validation_f1 - default_validation_f1:+.4f}")

sweep_t = [t for t, _ in f1_at_threshold]
sweep_f1 = [f for _, f in f1_at_threshold]
plt.figure(figsize=(7, 4))
plt.plot(sweep_t, sweep_f1, color="tab:blue")
plt.axvline(tuned_threshold, linestyle="--", color="tab:green", label=f"tuned = {tuned_threshold:.3f}")
plt.axvline(DEFAULT_THRESHOLD, linestyle=":", color="tab:gray", label=f"default = {DEFAULT_THRESHOLD:.2f}")
plt.xlabel("Decision threshold")
plt.ylabel("Validation F1")
plt.title("F1 vs decision threshold (validation set)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Evaluate the tuned neural network

Both operating points are reported: the **tuned** threshold (the primary result, chosen for best F1)
and the **default 0.50** threshold (for reference and fair comparison with models fixed at 0.5). The
confusion matrix, per-attack breakdown, and summary below use the tuned threshold.

In [ ]:
def evaluate_binary_classifier(name, y_true, scores, threshold, prediction_time):
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    result = {
        "dataset": name,
        "threshold": round(float(threshold), 4),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp),
        "fpr": fp / (fp + tn),
        "fnr": fn / (fn + tp),
        "roc_auc": roc_auc_score(y_true, scores),
        "pr_auc": average_precision_score(y_true, scores),
        "alerts": int(y_pred.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "prediction_time_sec": prediction_time
    }

    return result, y_pred

start_test = time.perf_counter()
test_scores = mlp_pipeline.predict_proba(X_test)[:, 1]
test_prediction_time = time.perf_counter() - start_test

rows = []
for label, thr in [("Tuned", tuned_threshold), ("Default 0.50", DEFAULT_THRESHOLD)]:
    val_row, val_pred = evaluate_binary_classifier(f"Validation ({label})", y_validation, validation_scores, thr, np.nan)
    test_row, tst_pred = evaluate_binary_classifier(f"Test ({label})", y_test, test_scores, thr, test_prediction_time)
    rows.extend([val_row, test_row])
    if label == "Tuned":
        validation_pred = val_pred
        test_pred = tst_pred

metrics_df = pd.DataFrame(rows)
display(metrics_df.round(4))

tuned_test_f1 = metrics_df.loc[metrics_df["dataset"] == "Test (Tuned)", "f1"].iloc[0]
default_test_f1 = metrics_df.loc[metrics_df["dataset"] == "Test (Default 0.50)", "f1"].iloc[0]
print(f"Test F1 at tuned threshold:   {tuned_test_f1:.4f}")
print(f"Test F1 at default 0.50:      {default_test_f1:.4f}")

### Confusion matrices

Evaluated on the untouched validation and test sets at the **tuned threshold**. Hyperparameter tuning
(model selection, on PR-AUC) and threshold choice (operating point, on F1) are kept as separate steps.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(title)
    plt.colorbar()
    plt.xticks([0, 1], ["Benign", "Attack"])
    plt.yticks([0, 1], ["Benign", "Attack"])

    for i in range(2):
        for j in range(2):
            plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center")

    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(y_validation, validation_pred, "Tuned MLP Validation Confusion Matrix")
plot_confusion_matrix(y_test, test_pred, "Tuned MLP Test Confusion Matrix")

### Test classification report

In [ ]:
report = classification_report(
    y_test,
    test_pred,
    target_names=["Benign", "Attack"],
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T
display(report_df.round(4))

### Results by traffic type

Because the binary target hides which attacks the network catches, this breaks the test predictions
down by the original `attack_type`, showing how the correct-classification rate varies across benign
traffic and each of the five attack families.

In [ ]:
type_results = test_df[[ATTACK_COL]].copy()
type_results["y_true"] = y_test
type_results["y_pred"] = test_pred
type_results["correct"] = type_results["y_true"] == type_results["y_pred"]

result_by_type = type_results.groupby(ATTACK_COL).agg(total=("correct", "size"), correct=("correct", "sum"))
result_by_type["incorrect"] = result_by_type["total"] - result_by_type["correct"]
result_by_type["correct_rate"] = result_by_type["correct"] / result_by_type["total"] * 100
result_by_type["incorrect_rate"] = 100 - result_by_type["correct_rate"]

type_order = ["Benign", "DDoS-HTTP Flood", "DoS-HTTP Flood", "DNS Spoofing", "Brute Force", "XSS"]
result_by_type = result_by_type.reindex(type_order).dropna()

display(result_by_type[["total", "correct", "incorrect", "correct_rate"]].round(2))

plot_data = result_by_type[["correct_rate", "incorrect_rate"]]
ax = plot_data.plot(kind="barh", stacked=True, figsize=(11, 6))

for i, (_, row) in enumerate(result_by_type.iterrows()):
    count_text = f'Correct: {int(row["correct"]):,}   Incorrect: {int(row["incorrect"]):,}'
    ax.text(102, i, count_text, va="center", fontsize=9)

ax.set_xlim(0, 145)
ax.set_xlabel("Percentage of flows")
ax.set_ylabel("Traffic type")
ax.set_title("Correct and Incorrect Tuned MLP Classification by Traffic Type")
ax.legend(["Correct", "Incorrect"], loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

### ROC and precision-recall curves

In [ ]:
def plot_roc_curve(y_true, scores, title):
    fpr, tpr, _ = roc_curve(y_true, scores)
    auc_value = roc_auc_score(y_true, scores)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {auc_value:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_precision_recall_curve(y_true, scores, title):
    precision, recall, _ = precision_recall_curve(y_true, scores)
    pr_auc = average_precision_score(y_true, scores)

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

plot_roc_curve(y_test, test_scores, "Tuned MLP Test ROC Curve")
plot_precision_recall_curve(y_test, test_scores, "Tuned MLP Test Precision-Recall Curve")

### Training loss curve

The neural-network-specific diagnostic. `loss_curve_` is the training loss at each epoch and
`validation_scores_` is the internal early-stopping validation score. If the training loss keeps
falling while the validation score plateaus or drops, the network is starting to overfit and early
stopping is doing its job.

In [ ]:
best_mlp = mlp_pipeline.named_steps["model"]

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(best_mlp.loss_curve_, color="tab:blue", label="Training loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training loss", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

if hasattr(best_mlp, "validation_scores_") and best_mlp.validation_scores_ is not None:
    ax2 = ax1.twinx()
    ax2.plot(best_mlp.validation_scores_, color="tab:orange", label="Validation score")
    ax2.set_ylabel("Internal validation score", color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")

plt.title("Tuned MLP Training History")
fig.tight_layout()
plt.show()

print("Epochs trained:", best_mlp.n_iter_)
print("Final training loss:", round(best_mlp.loss_curve_[-1], 6))

## Save the tuned results

Results are written to `reports/phase3/` using the same file-naming convention as the other Phase 3
models, so the cascade notebook and the final comparison table can pick them up automatically.

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "reports" / "phase3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

metrics_output = metrics_df.copy()
metrics_output.insert(0, "model", "Neural Network Tuned")
metrics_output["search_time_sec"] = search_time
metrics_output["refit_time_sec"] = mlp_search.refit_time_
metrics_output.to_csv(RESULTS_DIR / "neural_network_tuned_metrics.csv", index=False)

cv_results_df.to_csv(RESULTS_DIR / "neural_network_random_search_results.csv", index=False)

best_parameters = {
    "best_cv_pr_auc": float(mlp_search.best_score_),
    "search_time_sec": search_time,
    "refit_time_sec": mlp_search.refit_time_,
    "n_iter": N_ITER,
    "cv_folds": cv.get_n_splits(),
    "default_threshold": DEFAULT_THRESHOLD,
    "tuned_threshold": float(tuned_threshold),
    "tuned_validation_f1": float(tuned_validation_f1),
    "max_benign_train": MAX_BENIGN_TRAIN,
    "attack_to_benign": ATTACK_TO_BENIGN,
    "epochs_trained": int(best_mlp.n_iter_),
    "best_params": {k: (list(v) if isinstance(v, tuple) else (float(v) if hasattr(v, "item") else v))
                    for k, v in mlp_search.best_params_.items()}
}

with open(RESULTS_DIR / "neural_network_best_parameters.json", "w") as f:
    json.dump(best_parameters, f, indent=2)

prediction_cols = [FLOW_KEY_COL, ATTACK_COL, BINARY_COL, TARGET_COL]
test_predictions = test_df[prediction_cols].copy()
test_predictions["attack_probability"] = test_scores
test_predictions["prediction"] = test_pred
test_predictions.to_csv(RESULTS_DIR / "neural_network_tuned_test_predictions.csv", index=False)

print("Saved tuned neural network results to:", RESULTS_DIR)

## Final tuned neural network summary

In [ ]:
test_row = metrics_df.loc[metrics_df["dataset"] == "Test (Tuned)"].iloc[0]

print("Final tuned neural network test results (at tuned threshold)")
print("-----------------------------------------------------------")
print(f"Best mean CV PR-AUC: {mlp_search.best_score_:.4f}")
print(f"Tuned threshold: {tuned_threshold:.3f}")
print(f"Accuracy: {test_row['accuracy']:.4f}")
print(f"Precision: {test_row['precision']:.4f}")
print(f"Recall: {test_row['recall']:.4f}")
print(f"F1-score: {test_row['f1']:.4f}")
print(f"ROC-AUC: {test_row['roc_auc']:.4f}")
print(f"PR-AUC: {test_row['pr_auc']:.4f}")
print(f"FPR: {test_row['fpr']:.4f}")
print(f"FNR: {test_row['fnr']:.4f}")
print(f"Search time: {search_time:.2f} seconds")
print(f"Best-model refit time: {mlp_search.refit_time_:.2f} seconds")
print(f"Test prediction time: {test_prediction_time:.2f} seconds")

print()
print("Best hyperparameters")
for name, value in mlp_search.best_params_.items():
    print(f"{name}: {value}")

print()
print("Correct classification rate by traffic type")
for attack_type in type_order:
    if attack_type in result_by_type.index:
        row = result_by_type.loc[attack_type]
        print(f"{attack_type}: {row['correct_rate']:.2f}%")